In [3]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 63.4 MB/s  0:00:02m0:00:0100:01


In [1]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

print("Loading data...")
data = torch.load("./chemBERTa_dataset.pt", weights_only=False)
print("Data loaded")

print("Converting tensors...")
X = data["X"].numpy()
y = np.array(data["y"])
print("Conversion done")

print("Splitting dataset...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Split done")

print("Training XGBoost...")

models = []

for i in tqdm(range(y_train.shape[1]), desc="Training classifiers"):
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",   # required
        device="cuda",        # 👈 uses GPU
        verbosity=0,
        n_jobs=-1
    )

    model.fit(X_train, y_train[:, i])
    models.append(model)

print("Training complete")

from sklearn.metrics import f1_score, roc_auc_score

print("Predicting...")
probs = np.column_stack([model.predict_proba(X_test)[:, 1] for model in models])

y_pred = (probs > 0.5).astype(int)

print("Prediction done")

f1 = f1_score(y_test, y_pred, average="micro")
print("F1 Score:", f1)

try:
    auc = roc_auc_score(y_test, probs, average="micro")
    print("ROC-AUC Score:", auc)
except Exception as e:
    print("AUC could not be computed:", e)

Loading data...
Data loaded
Converting tensors...
Conversion done
Splitting dataset...
Split done
Training XGBoost...


Training classifiers: 100%|██████████| 963/963 [46:48<00:00,  2.92s/it]


Training complete
Predicting...
Prediction done
F1 Score: 0.04020889597090107
ROC-AUC Score: 0.8338714478327147
